# Day 3: Supervised ML — Linear Regression

<a target="_blank" href="https://colab.research.google.com/github/LuWidme/uk259/blob/main/demos/05_SupervisedML_LinearRegression.ipynb">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

**Duration:** 1.5–2 hours  
**Prerequisites:** Python basics, NumPy, Pandas, data preprocessing  

**Learning Objectives:**
- Understand what linear regression is and when to use it
- Build regression models with scikit-learn and statsmodels
- Evaluate models using MAE, RMSE, and R²
- Analyze residuals to check model assumptions
- Understand Simpson's Paradox

**Datasets Used:** Penguins (seaborn), Tips (seaborn), Melbourne Housing (`datasets/melb_data.csv`)

---

## What is Linear Regression?

Linear regression predicts a **continuous numerical value** (unlike classification, which predicts categories).

**Examples:**
- Predict **house price** from size, location, rooms
- Predict **tip amount** from bill total
- Predict **body mass** from flipper length

**How it works:** Find the "best-fit" line through the data.

**Simple linear regression** (one feature): `y = mx + b`
- `m` = slope (how much y changes when x increases by 1)
- `b` = intercept (value of y when x = 0)

**Multiple linear regression** (many features): `y = b₀ + b₁x₁ + b₂x₂ + ... + bₙxₙ`

![img](https://github.com/LuWidme/uk259/blob/main/img/1_N1-K-A43_98pYZ27fnupDA%20(1).jpg?raw=1)

---

## Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

sns.set_theme()
print("✓ Libraries imported successfully!")

---

## Part 1: Exploring the Data

We'll start with the **Palmer Penguins** dataset — measurements of penguins from three species.

In [ ]:
penguins = sns.load_dataset('penguins')
print(f"Dataset: {penguins.shape[0]} penguins, {penguins.shape[1]} columns")
print(f"Species: {list(penguins['species'].unique())}")
penguins.head()

In [ ]:
# Pairplot: visualize relationships between all numerical features
sns.pairplot(data=penguins, hue='species')
plt.suptitle('Penguin Feature Relationships by Species', y=1.02)
plt.show()

### Task 1: Visualize a Linear Relationship

Use `sns.lmplot` to show the correlation between `bill_length_mm` (x-axis) and `bill_depth_mm` (y-axis). The function automatically fits a regression line.

**Hint:** `sns.lmplot(data=penguins, x='bill_length_mm', y='bill_depth_mm')`

Do you notice anything surprising about the trend?

In [ ]:
# TODO: Plot bill_length_mm vs bill_depth_mm with a regression line
# Hint: sns.lmplot(data=penguins, x='...', y='...')

g = None  # TODO: create the lmplot

plt.show()

### Simpson's Paradox

The overall trend shows a **negative** correlation (longer bill → shallower depth). But is this real?

This is **Simpson's Paradox**: a trend in aggregated data **reverses** when you look at subgroups.

Let's separate by species:

In [ ]:
g = sns.lmplot(
    data=penguins,
    x='bill_length_mm',
    y='bill_depth_mm',
    hue='species',
    height=6,
    aspect=1.5
)

g.set_axis_labels('Bill length (mm)', 'Bill depth (mm)')
plt.suptitle("Simpson's Paradox: Trend Reverses Within Species!", y=1.02)
plt.show()

print("Key Insight:")
print("Within each species, bill length and depth are POSITIVELY correlated!")
print("The overall negative trend was caused by differences between species.")
print("\nLesson: Always explore subgroups before drawing conclusions.")

---

## Part 2: Linear Regression with Scikit-Learn

Let's predict penguin **body mass** from **flipper length** using scikit-learn.

In [ ]:
# Prepare data
penguins_clean = penguins.dropna(subset=['flipper_length_mm', 'body_mass_g'])

X = penguins_clean[['flipper_length_mm']].values  # Features (must be 2D)
y = penguins_clean['body_mass_g'].values           # Target

# Split into train/test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Training samples: {len(X_train)}")
print(f"Test samples: {len(X_test)}")

# Train the model
model = LinearRegression()
model.fit(X_train, y_train)

print(f"\nLearned equation:")
print(f"  body_mass = {model.coef_[0]:.2f} × flipper_length + {model.intercept_:.2f}")
print(f"\nInterpretation: For every 1mm increase in flipper length,")
print(f"  body mass increases by ~{model.coef_[0]:.0f}g")

In [ ]:
# Visualize predictions
y_pred = model.predict(X_test)

plt.figure(figsize=(10, 6))
plt.scatter(X_train, y_train, alpha=0.5, label='Training data', color='blue')
plt.scatter(X_test, y_test, alpha=0.5, label='Test data', color='green')

# Regression line
X_line = np.linspace(X.min(), X.max(), 100).reshape(-1, 1)
plt.plot(X_line, model.predict(X_line), 'r-', linewidth=2, label='Regression line')

plt.xlabel('Flipper Length (mm)')
plt.ylabel('Body Mass (g)')
plt.title('Linear Regression: Predicting Penguin Body Mass')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

---

## Part 3: Evaluation Metrics

How do we know if our model is good? We need quantitative metrics.

| Metric | What it measures | Interpretation |
|--------|-----------------|----------------|
| **MAE** | Average absolute error | "On average, we're off by X grams" |
| **RMSE** | Root of average squared error | Penalizes large errors more |
| **R²** | Proportion of variance explained | 1.0 = perfect, 0 = useless |

In [ ]:
# Calculate metrics
mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, y_pred)
r2_train = r2_score(y_train, model.predict(X_train))

print("=" * 50)
print("MODEL EVALUATION")
print("=" * 50)
print(f"\nMean Absolute Error (MAE):  {mae:.2f} g")
print(f"Root Mean Squared Error:    {rmse:.2f} g")
print(f"R² Score (test):            {r2:.4f}")
print(f"R² Score (train):           {r2_train:.4f}")

print(f"\nInterpretation:")
print(f"- The model explains {r2*100:.1f}% of the variance in body mass")
print(f"- Predictions are off by ~{mae:.0f}g on average")
if abs(r2_train - r2) < 0.05:
    print("- Small train/test gap → no overfitting")

---

## Part 4: Residual Analysis

**Residuals** = actual values − predicted values

Analyzing residuals tells us if the model's assumptions hold:
- Residuals should be **randomly scattered** (no pattern)
- Residuals should have **constant spread** (homoscedasticity)
- Residuals should be **normally distributed**

In [ ]:
from scipy import stats

residuals = y_test - y_pred

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Plot 1: Residuals vs Predicted
axes[0, 0].scatter(y_pred, residuals, alpha=0.6)
axes[0, 0].axhline(y=0, color='r', linestyle='--', linewidth=2)
axes[0, 0].set_xlabel('Predicted Values')
axes[0, 0].set_ylabel('Residuals')
axes[0, 0].set_title('Residuals vs Predicted')
axes[0, 0].grid(True, alpha=0.3)

# Plot 2: Histogram of Residuals
axes[0, 1].hist(residuals, bins=20, edgecolor='black', alpha=0.7)
axes[0, 1].set_xlabel('Residuals')
axes[0, 1].set_ylabel('Frequency')
axes[0, 1].set_title('Distribution of Residuals')
axes[0, 1].grid(True, alpha=0.3)

# Plot 3: Q-Q Plot
stats.probplot(residuals, dist='norm', plot=axes[1, 0])
axes[1, 0].set_title('Q-Q Plot')
axes[1, 0].grid(True, alpha=0.3)

# Plot 4: Actual vs Predicted
axes[1, 1].scatter(y_test, y_pred, alpha=0.6)
axes[1, 1].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()],
                'r--', lw=2, label='Perfect prediction')
axes[1, 1].set_xlabel('Actual Values')
axes[1, 1].set_ylabel('Predicted Values')
axes[1, 1].set_title('Actual vs Predicted')
axes[1, 1].legend()
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("How to read these plots:")
print("1. Residuals vs Predicted: Points should be randomly scattered around y=0")
print("2. Histogram: Should look like a bell curve (normal distribution)")
print("3. Q-Q Plot: Points should follow the diagonal line")
print("4. Actual vs Predicted: Points should follow the red dashed line")

---

## Part 5: Multiple Regression with Statsmodels

For regression with **multiple input variables**, the `statsmodels` library gives us a detailed statistical summary including p-values and confidence intervals.

We'll use the **Tips** dataset to predict tip amount from bill total, party size, and smoking status.

In [ ]:
from statsmodels.formula.api import ols

tips = sns.load_dataset('tips')
print(f"Tips dataset: {tips.shape[0]} meals")
tips.head()

In [ ]:
# Fit a multiple regression model
formula = 'tip ~ total_bill + smoker + size'
result = ols(formula=formula, data=tips).fit()
result.summary()

### Reading the OLS Summary

**Key values to look at:**

1. **R²** (top right): How much variance the model explains (0–1, higher is better)
2. **Adj. R²**: R² adjusted for the number of features (use this when comparing models)
3. **Coefficients table:**
   - `coef`: How much `tip` changes when this feature increases by 1
   - `P>|t|` (p-value): Is this feature statistically significant?
     - **p < 0.05** → significant (keep this feature)
     - **p > 0.05** → not significant (consider dropping it)

**Example interpretation:** If `total_bill` has coef = 0.094, it means: for every €1 increase in the bill, the tip increases by ~€0.09.

### Visualizing the Simple Regression Line

In [ ]:
# Simple regression: tip ~ total_bill
result_simple = ols(formula='tip ~ total_bill', data=tips).fit()

m = result_simple.params['total_bill']
c = result_simple.params['Intercept']

plt.figure(figsize=(10, 6))
plt.scatter(tips['total_bill'], tips['tip'], alpha=0.5)
x_line = np.linspace(tips['total_bill'].min(), tips['total_bill'].max(), 100)
plt.plot(x_line, m * x_line + c, 'r-', linewidth=2, label=f'y = {m:.3f}x + {c:.3f}')
plt.xlabel('Total Bill (€)')
plt.ylabel('Tip (€)')
plt.title('Predicting Tips from Total Bill')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

print(f"R² = {result_simple.rsquared:.4f}")
print(f"For every €1 increase in bill, the tip increases by ~€{m:.2f}")

---

## Part 6: Exercises

### Exercise 1: Melbourne Housing Price Prediction

**Scenario:** Predict house prices using the Melbourne housing dataset.

**Tasks:**
1. Load `datasets/melb_data.csv` and explore it
2. Select numerical features and handle missing values
3. Build a linear regression model with scikit-learn
4. Evaluate using MAE, RMSE, and R²
5. Create residual plots

**Hints:**
- Good features to start with: `Rooms`, `Distance`, `Bathroom`, `Landsize`, `BuildingArea`
- Use `df.dropna()` or `df.fillna(df.median())` for missing values
- Target variable: `Price`

In [ ]:
# Step 1: Load and explore the Melbourne housing data
melb = pd.read_csv('../datasets/melb_data.csv')
print(f"Dataset: {melb.shape[0]} houses, {melb.shape[1]} columns")
print(f"\nColumns: {list(melb.columns)}")
print(f"\nMissing values (top 5):")
print(melb.isnull().sum().sort_values(ascending=False).head())
melb.head()

In [ ]:
# Step 2: Select features and prepare data

# TODO: Choose features for the model
feature_cols = None  # TODO: e.g., ['Rooms', 'Distance', 'Bathroom', 'Landsize', 'BuildingArea']

# TODO: Create feature matrix X and target vector y
# Hint: Drop rows where any selected feature or Price is NaN
X_melb = None  # TODO
y_melb = None  # TODO: melb['Price'] (after dropping NaN)

# TODO: Split into train/test
# Hint: train_test_split(X_melb, y_melb, test_size=0.2, random_state=42)

In [ ]:
# Step 3 & 4: Train model and evaluate

# TODO: Create and train a LinearRegression model

# TODO: Make predictions on test set

# TODO: Calculate and print MAE, RMSE, R²

# TODO: Print the coefficients and their feature names
# Hint: model.coef_ gives the coefficients, pair with feature_cols

In [ ]:
# Step 5: Create residual plots

# TODO: Calculate residuals (y_test - y_pred)
# TODO: Create at least 2 diagnostic plots:
#   - Residuals vs Predicted
#   - Actual vs Predicted
# Do the residuals look random? Any patterns?

---

### Exercise 2: Feature Selection with Statsmodels

**Task:** Use statsmodels OLS to find which features are statistically significant predictors of tip amount.

1. Start with the formula: `tip ~ total_bill + size + smoker + sex + day + time`
2. Check p-values for each feature
3. Remove non-significant features (p > 0.05) one at a time
4. Compare R² of the full model vs. the simplified model

In [ ]:
# TODO: Fit the full model with all features
full_formula = None  # TODO: 'tip ~ total_bill + size + smoker + sex + day + time'

# TODO: result_full = ols(formula=full_formula, data=tips).fit()
# TODO: print(result_full.summary())

# TODO: Which features have p > 0.05? Remove them.

# TODO: Fit a simplified model without non-significant features
# TODO: Compare R² of full vs simplified model

---

### Exercise 3: Penguin Body Mass Prediction (Multiple Features)

**Task:** Improve the single-feature model from Part 2 by adding more features.

1. Use multiple numerical features: `flipper_length_mm`, `bill_length_mm`, `bill_depth_mm`
2. Train a new model and compare R² with the single-feature model
3. Does adding features improve the prediction?

In [ ]:
# TODO: Prepare multi-feature data from penguins_clean
# Features: flipper_length_mm, bill_length_mm, bill_depth_mm
# Target: body_mass_g

X_multi = None  # TODO: penguins_clean[['flipper_length_mm', 'bill_length_mm', 'bill_depth_mm']].values
y_multi = None  # TODO: penguins_clean['body_mass_g'].values

# TODO: Split, train, and evaluate
# TODO: Compare R² with the single-feature model from Part 2
# (single-feature R² was printed above)

# TODO: Print which features are most important (largest coefficients)

---

## Summary

In this notebook you learned:

✓ What linear regression is and how to interpret the equation  
✓ Simpson's Paradox — aggregated trends can be misleading  
✓ Building regression models with scikit-learn  
✓ Evaluation metrics: MAE, RMSE, R²  
✓ Residual analysis to check model assumptions  
✓ Multiple regression with statsmodels OLS  
✓ Reading p-values to assess feature significance  

### Key Takeaways

1. **Always visualize first** — check for linearity before fitting a line
2. **Simpson's Paradox** — always explore subgroups in your data
3. **R² is not everything** — check residual plots too
4. **p-values help feature selection** — drop features with p > 0.05
5. **More features aren't always better** — compare adjusted R²

### What's Next?

- **Polynomial regression:** Fit curves, not just lines
- **Regularization:** Ridge and Lasso regression to prevent overfitting
- **Neural Networks:** Learn complex, non-linear patterns

---